# 2027 NFL Big Data Bowl — Combine Drill Position Maps

This notebook maps out **every recorded `(x, y)` position** across all **62 distinct Combine drills** (`drill_type`, `drill_name`) in `combine_tracking.parquet` (`463,189` total 10 Hz frames across `6,310` drill attempts and `510` prospects).

- **Player positions (`PLAYER`)** are plotted as a scatter of every recorded frame, colored by instantaneous speed `s` (yd/s).
- **Ball positions (`BALL`)**, where tracked (e.g., WR/TE route drills), are overlaid in **orange**.
- Field markings show regulation yard lines (`0–120` yd length, `0–53.33` yd width).

In [ ]:
import math
from pathlib import Path
import polars as pl
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = Path("../combine_tracking.parquet")
if not DATA_PATH.exists():
    DATA_PATH = Path("c:/#NFL/combine_tracking.parquet")

df = pl.read_parquet(DATA_PATH)
print(f"Loaded {len(df):,} frames across {df['event_id'].n_unique():,} events and {df['nfl_id'].n_unique()} players.")

## 1. Summary of All 62 Combine Drills

In [ ]:
summary = (
    df.group_by(["drill_type", "drill_name"])
    .agg([
        pl.len().alias("n_frames"),
        pl.col("event_id").n_unique().alias("n_events"),
        pl.col("nfl_id").n_unique().alias("n_players"),
        (pl.col("entity_type") == "BALL").any().alias("has_ball"),
        pl.col("x").min().round(1).alias("x_min"),
        pl.col("x").max().round(1).alias("x_max"),
        pl.col("y").min().round(1).alias("y_min"),
        pl.col("y").max().round(1).alias("y_max"),
    ])
    .sort(["drill_type", "drill_name"])
)
pl.Config.set_tbl_rows(100)
print(summary)

## 2. Field Helper & Per-Drill Position Mapper
Each subplot renders **every recorded `(x, y)` point** for that drill, zoomed to the drill's active field region (with 2-yard padding) and overlaid with regulation 5-yard / 10-yard lines.

In [ ]:
def draw_field_lines(ax, x_min, x_max, y_min, y_max):
    """Draw regulation NFL yard lines and sidelines within the visible bounds."""
    ax.set_facecolor("#f8fafc")
    ax.axhline(0.0, color="#334155", lw=1.2, ls="-", alpha=0.6)
    ax.axhline(53.33, color="#334155", lw=1.2, ls="-", alpha=0.6)
    for x_line in range(0, 121, 5):
        if x_min <= x_line <= x_max:
            is_major = (x_line % 10 == 0)
            ax.axvline(
                x_line,
                color="#94a3b8" if is_major else "#cbd5e1",
                lw=1.0 if is_major else 0.5,
                ls="-" if is_major else "--",
                alpha=0.7,
                zorder=1,
            )


def plot_drill_group(df_all: pl.DataFrame, drill_type: str, ncols: int = 3):
    """Plot every recorded (x, y) position for every drill_name in a drill_type."""
    sub = df_all.filter(pl.col("drill_type") == drill_type)
    drill_names = sorted(sub["drill_name"].unique().to_list())
    n = len(drill_names)
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(6.2 * ncols, 4.5 * nrows), squeeze=False)
    fig.suptitle(f"{drill_type} — Every Recorded Position ({n} drills)", fontsize=15, fontweight="bold", y=1.01)

    for idx, dname in enumerate(drill_names):
        ax = axes[idx // ncols][idx % ncols]
        d_df = sub.filter(pl.col("drill_name") == dname)
        p_df = d_df.filter(pl.col("entity_type") == "PLAYER")
        b_df = d_df.filter(pl.col("entity_type") == "BALL")

        x_min, x_max = float(d_df["x"].min()) - 2.0, float(d_df["x"].max()) + 2.0
        y_min, y_max = float(d_df["y"].min()) - 2.0, float(d_df["y"].max()) + 2.0
        draw_field_lines(ax, x_min, x_max, y_min, y_max)

        if len(p_df) > 0:
            ax.scatter(
                p_df["x"].to_numpy(),
                p_df["y"].to_numpy(),
                c=p_df["s"].to_numpy(),
                cmap="viridis",
                vmin=0.0,
                vmax=10.5,
                s=3,
                alpha=0.25,
                edgecolors="none",
                zorder=2,
                label=f"PLAYER ({len(p_df):,} pts)",
            )
        if len(b_df) > 0:
            ax.scatter(
                b_df["x"].to_numpy(),
                b_df["y"].to_numpy(),
                color="#f97316",
                s=5,
                alpha=0.35,
                edgecolors="none",
                zorder=3,
                label=f"BALL ({len(b_df):,} pts)",
            )

        n_ev = d_df["event_id"].n_unique()
        n_pl = d_df["nfl_id"].n_unique()
        ax.set_title(f"{dname}\n({n_pl} players, {n_ev} events, {len(d_df):,} frames)", fontsize=10, fontweight="bold")
        ax.set_xlim(x_min, x_max)
        ax.set_ylim(y_min, y_max)
        ax.set_xlabel("Field X (yd)", fontsize=8)
        ax.set_ylabel("Field Y (yd)", fontsize=8)
        ax.set_aspect("equal", adjustable="datalim")
        if len(b_df) > 0:
            ax.legend(loc="upper right", fontsize=7, markerscale=2.5)

    for j in range(n, nrows * ncols):
        axes[j // ncols][j % ncols].axis("off")

    plt.tight_layout()
    plt.show()

## 3. Plot Every Drill by `drill_type`
Iterates over all 9 `drill_type` categories (`FORTY_YARD_DASH`, `SHORT_SHUTTLE`, `THREE_CONE_DRILL`, `SKILL_DRILLS_DB`, `SKILL_DRILLS_DL`, `SKILL_DRILLS_LB`, `SKILL_DRILLS_OL`, `SKILL_DRILLS_TE`, `SKILL_DRILLS_WR`) and maps every recorded position.

In [ ]:
drill_types = sorted(df["drill_type"].unique().to_list())
for dt in drill_types:
    plot_drill_group(df, dt, ncols=3)